[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/11_sliding_window_solution.ipynb)

# 🔴 Solution: Sliding Window Attention

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `11_sliding_window.ipynb` first.

---
Implement **sliding-window attention**: causal attention where query $i$ may
only look at the `window_size` most recent positions, itself included.

$$\text{out}_i = \sum_{j \in \mathcal{W}(i)} \alpha_{ij} v_j,
\qquad \mathcal{W}(i) = \{\, j : i - W < j \le i \,\}$$

$$\alpha_{i\cdot} = \mathrm{softmax}_{j \in \mathcal{W}(i)}\!\left(\frac{q_i \cdot k_j}{\sqrt{d}}\right)$$

So `window_size=1` means "attend to yourself only", `window_size=3` means
"yourself plus the two previous tokens", and any `window_size >= seq` is ordinary
causal attention.

### Rules
- Plain function — no `nnx.Module`, no parameters
- Banned: `jax.nn.dot_product_attention` (its `local_window_size` /
  `is_causal` flags do exactly this), `nnx.MultiHeadAttention`
- Signature: `sliding_window_attention(Q, K, V, window_size)`
- Shapes: `Q, K` are `(..., seq, d)` and `V` is `(..., seq, d_v)`, with **any**
  number of leading axes — `(seq, d)`, `(B, seq, D)` and `(B, H, seq, Dh)` must all
  work through broadcasting
- Scale by `1 / sqrt(d)` where `d = Q.shape[-1]`
- `window_size >= 1` is a Python `int`; masked positions get `-inf` before the
  softmax, never a post-softmax zeroing
- Return shape `(..., seq, d_v)`

### Why a window is enough: receptive field through depth
One layer of sliding-window attention only mixes information across $W$
positions. Stack $L$ of them and the receptive field is
$\approx L \cdot (W - 1) + 1$, because layer 2 reads tokens that already
absorbed their own windows at layer 1 — the same argument as stacked
convolutions. Mistral-7B ships $W = 4096$ over 32 layers, giving a theoretical
reach past 130k tokens while every layer only ever computes an
$O(seq \cdot W)$ band instead of an $O(seq^2)$ square.

The decode-time consequence is bigger than the FLOP saving: keys and values
older than $W$ can never be read again, so the KV cache becomes a **rolling
buffer of fixed size $W$**. Cache memory stops growing with context length —
that is what makes a 100k-token conversation fit on one GPU.

What the receptive-field argument does *not* say is that a distant token has
the same *influence* as a nearby one. Information reaches far, but it is
re-mixed and diluted at every hop, which is why long-context models interleave a
few full-attention layers among the windowed ones instead of trusting depth
alone.

### The traps
- **Post-softmax masking is wrong.** Zeroing weights after the softmax leaves
  the denominator polluted by out-of-window scores, so the surviving weights no
  longer sum to 1. Mask the *scores*.
- **A fully masked row produces NaN.** `softmax([-inf, -inf, ...])` is
  $0/0$. Here the diagonal is always inside the window so every row has at
  least one live entry — which is exactly why `window_size` counts *inclusive*
  of the query position and must be at least 1.
- Prefer `jnp.where(mask, scores, -inf)` over `scores + (1 - mask) * -1e9`: at
  bf16 the magic constant saturates, and adding it to already-large scores can
  overflow.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def sliding_window_attention(Q, K, V, window_size):
    seq, d = Q.shape[-2], Q.shape[-1]

    # (..., seq, seq) — the leading batch/head axes ride along via broadcasting.
    scores = jnp.einsum("...td,...sd->...ts", Q, K) / jnp.sqrt(jnp.float32(d))

    i = jnp.arange(seq)[:, None]      # query position
    j = jnp.arange(seq)[None, :]      # key position
    band = (j <= i) & (i - j < window_size)     # causal AND within the window

    # Mask the SCORES, so the softmax denominator only sees live positions.
    scores = jnp.where(band, scores, -jnp.inf)
    weights = jax.nn.softmax(scores, axis=-1)

    return jnp.einsum("...ts,...se->...te", weights, V)

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

seq = 6
# Q = 0 makes every in-window score equal, so the weights are uniform and the
# output at position i is just the mean of V over its window.
Q = jnp.zeros((seq, 4))
K = jnp.zeros((seq, 4))
V = jnp.arange(seq, dtype=jnp.float32)[:, None]

for w in (1, 2, 3, seq):
    out = sliding_window_attention(Q, K, V, w)
    print(f"W={w}: {out[:, 0]}")
print("\n-> row i is the running mean of the last W values: the band in action")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("sliding_window")